# p2 — IRT Fitting Pipeline

**Purpose:** Fit a 2PL IRT model for every `(architecture, cohort)` group and write per-subtask difficulty and discrimination parameters to `data_irt/`.

**How it works:**
1. For a given `(ARCH, COHORT)` pair, load the subset of models from `data_raw_clean/models.parquet` and merge with the deduplicated benchmark scores
2. Build a `ResponseMatrix` (models × subtasks) and fit `XBetaTwoPL` via MLE
3. Save the fitted discrimination and difficulty vectors as rows in `data_irt/{BENCH}/{BENCH}_discriminations_2pl.csv` and `_difficulties_2pl.csv`, tagged with `cohort` and `arch` columns

The CSV format allows incremental runs — if a `(cohort, arch)` row already exists it is updated in place; otherwise a new row is appended.

**Input:** `data_raw_clean/models.parquet`, `data_raw_clean/{BENCH}_scores_unique.parquet`  
**Output:** `data_irt/{BENCH}/{BENCH}_discriminations_2pl.csv`, `data_irt/{BENCH}/{BENCH}_difficulties_2pl.csv`

## Config

Set `BENCH` to the benchmark you want to fit. `ARCHS` and `COHORTS` define the full grid; remove entries to skip specific groups.

In [1]:
BENCH = "bbh"   # one of: bbh | math | gpqa | musr

ARCHS = [
    "LlamaForCausalLM",
    "Qwen2ForCausalLM",
    "MistralForCausalLM",
    "Gemma2ForCausalLM",
]

COHORTS = ["Q2-2024", "Q3-2024", "Q4-2024", "Q1-2025"]

## Subtask Definitions

These lists must match the column names in `_scores_unique.parquet` produced by `p1_model_scores_collect.ipynb`. They also determine the column order in the output CSVs.

In [2]:
import os

import pandas as pd
from torch_measure.data import ResponseMatrix
from torch_measure_ext.beta_twopl import XBetaTwoPL

BBH_ITEMS = [
    "boolean_expressions", "causal_judgement", "date_understanding", "disambiguation_qa",
    "formal_fallacies", "geometric_shapes", "hyperbaton",
    "logical_deduction_5obj", "logical_deduction_7obj", "logical_deduction_3obj",
    "movie_recommendation", "navigate", "object_counting", "penguins_in_table",
    "reasoning_colored_obj", "ruin_names", "salient_translation", "snarks",
    "sports_understanding", "temporal_sequences",
    "tracking_5obj", "tracking_7obj", "tracking_3obj", "web_of_lies",
]

MATH_ITEMS = [
    "algebra", "counting_and_prob", "geometry", "intermediate_algebra",
    "num_theory", "prealgebra", "precalculus",
]

MUSR_ITEMS = ["murder_mysteries", "object_placements", "team_allocation"]

GPQA_ITEMS = ["diamond", "extended", "main"]

ITEMS_MAP = {
    "bbh":  BBH_ITEMS,
    "math": MATH_ITEMS,
    "gpqa": GPQA_ITEMS,
    "musr": MUSR_ITEMS,
}

items = ITEMS_MAP[BENCH]
print(f"Benchmark: {BENCH} — {len(items)} subtasks: {items}")

Benchmark: bbh — 24 subtasks: ['boolean_expressions', 'causal_judgement', 'date_understanding', 'disambiguation_qa', 'formal_fallacies', 'geometric_shapes', 'hyperbaton', 'logical_deduction_5obj', 'logical_deduction_7obj', 'logical_deduction_3obj', 'movie_recommendation', 'navigate', 'object_counting', 'penguins_in_table', 'reasoning_colored_obj', 'ruin_names', 'salient_translation', 'snarks', 'sports_understanding', 'temporal_sequences', 'tracking_5obj', 'tracking_7obj', 'tracking_3obj', 'web_of_lies']


## Helper Functions

### `read_responses(arch, cohort)`
Filters `models.parquet` to the target `(arch, cohort)` slice, merges with benchmark scores, and returns a `ResponseMatrix` (the input format expected by `torch_measure`).

### `save(tensor, name, cohort, arch)`
Persists a 1-D parameter tensor (one value per subtask) as a row in the output CSV. The CSV is keyed on `(cohort, arch)`: existing rows are updated in place, new combinations are appended. This makes it safe to re-run the pipeline for a subset of groups without clobbering other rows.

In [7]:
def read_responses(arch: str, cohort: str) -> ResponseMatrix:
    """Load and merge model metadata + scores for a single (arch, cohort) group."""
    meta = pd.read_parquet("../data_raw_clean/models.parquet")[
        ["fullname", "Architecture", "#Params (B)", "cohort"]
    ]
    meta = meta.drop_duplicates(subset="fullname", keep="first")
    meta = meta[(meta["Architecture"] == arch) & (meta["cohort"] == cohort)]

    scores = pd.read_parquet(f"../data_raw_clean/{BENCH}_scores_unique.parquet")

    merged = meta.merge(scores, left_on="fullname", right_on="model_name", how="inner")
    drop_cols = [c for c in ["fullname", "evaluation_date", "evaluation_date_ts",
                              "Architecture", "#Params (B)", "cohort"] if c in merged.columns]
    merged = merged.drop(columns=drop_cols).set_index("model_name")

    return ResponseMatrix.from_dataframe(merged)


def save(tensor, name: str, cohort: str, arch: str) -> None:
    """Write a parameter vector as a (cohort, arch) row in the output CSV."""
    nparr = tensor.detach().numpy()
    path  = f"../data_irt/{BENCH}/{BENCH}_{name}.csv"

    if os.path.exists(path):
        df   = pd.read_csv(path)
        mask = (df["cohort"] == cohort) & (df["arch"] == arch)
        if mask.any():
            df.loc[mask, items] = nparr
            print(f"{name}: Updated existing row  cohort='{cohort}'  arch='{arch}'")
        else:
            new_row = pd.DataFrame([nparr], columns=items)
            new_row["cohort"] = cohort
            new_row["arch"]   = arch
            df = pd.concat([df, new_row], ignore_index=True)
            print(f"{name}: Added new row          cohort='{cohort}'  arch='{arch}'")
    else:
        df = pd.DataFrame([nparr], columns=items)
        df["cohort"] = cohort
        df["arch"]   = arch

    df.to_csv(path, index=False)

## Fitting Pipeline

`pipe(arch, cohort)` runs the full sequence for one group:
1. Build the response matrix
2. Fit `XBetaTwoPL` (Beta-regularised 2PL) via MLE for up to `max_epochs=10000` iterations — the progress bar and convergence loss are printed by `verbose=True`
3. Print fitted parameter ranges as a sanity check
4. Save discrimination and difficulty vectors

Only discrimination and difficulty are saved (not ability) because ability estimates are not comparable across cohorts without additional alignment.

In [8]:
def pipe(arch: str, cohort: str) -> None:
    """Fit 2PL IRT for one (arch, cohort) group and save parameters."""
    print(f"\n{'='*60}")
    print(f"  arch={arch}  cohort={cohort}")
    print(f"{'='*60}")

    rm   = read_responses(arch, cohort)
    print(f"Response matrix: {rm.n_subjects} models × {rm.n_items} items")

    data = rm.data.to("cpu")
    mask = rm.observed_mask.to("cpu")

    model = XBetaTwoPL(n_subjects=rm.n_subjects, n_items=rm.n_items, device="cpu")
    model.fit(data, mask=mask, max_epochs=10000, verbose=True)

    abilities        = model.ability.detach().cpu()
    difficulties     = model.difficulty.detach().cpu()
    discriminations  = model.discrimination.detach().cpu()

    print(f"2PL abilities:        min = {abilities.min():.2f}, max = {abilities.max():.2f}")
    print(f"2PL difficulties:     min = {difficulties.min():.2f}, max = {difficulties.max():.2f}")
    print(f"2PL discriminations:  min = {discriminations.min():.2f}, max = {discriminations.max():.2f}, "
          f"mean = {discriminations.mean():.2f}")

    save(discriminations, "discriminations_2pl", cohort, arch)
    save(difficulties,    "difficulties_2pl",    cohort, arch)

## Run

Iterates over the full `ARCHS × COHORTS` grid defined in the Config cell. Each `(arch, cohort)` combination is independent — if a run fails, re-run just that group by setting `ARCHS` and `COHORTS` to single-element lists.

In [9]:
for arch in ARCHS:
    for cohort in COHORTS:
        pipe(arch, cohort)


  arch=LlamaForCausalLM  cohort=Q2-2024
Response matrix: 241 models × 24 items


MLE fitting:  19%|█▉        | 1895/10000 [00:02<00:11, 706.16it/s, loss=-0.880564]


2PL abilities:        min = -0.94, max = 0.28
2PL difficulties:     min = -0.93, max = 1.80
2PL discriminations:  min = 0.48, max = 6.24, mean = 1.67
discriminations_2pl: Updated existing row  cohort='Q2-2024'  arch='LlamaForCausalLM'
difficulties_2pl: Updated existing row  cohort='Q2-2024'  arch='LlamaForCausalLM'

  arch=LlamaForCausalLM  cohort=Q3-2024
Response matrix: 279 models × 24 items


MLE fitting:  17%|█▋        | 1715/10000 [00:02<00:12, 677.92it/s, loss=-0.897918]


2PL abilities:        min = -0.62, max = 0.89
2PL difficulties:     min = -0.44, max = 2.92
2PL discriminations:  min = 0.29, max = 5.92, mean = 1.74
discriminations_2pl: Updated existing row  cohort='Q3-2024'  arch='LlamaForCausalLM'
difficulties_2pl: Updated existing row  cohort='Q3-2024'  arch='LlamaForCausalLM'

  arch=LlamaForCausalLM  cohort=Q4-2024
Response matrix: 402 models × 24 items


...

2PL abilities:        min = -0.69, max = 0.17
2PL difficulties:     min = -1.42, max = 2.35
2PL discriminations:  min = 0.29, max = 3.56, mean = 1.42
discriminations_2pl: Updated existing row  cohort='Q3-2024'  arch='Gemma2ForCausalLM'
difficulties_2pl: Updated existing row  cohort='Q3-2024'  arch='Gemma2ForCausalLM'

  arch=Gemma2ForCausalLM  cohort=Q4-2024
Response matrix: 132 models × 24 items


MLE fitting:   3%|▎         | 259/10000 [00:00<00:11, 861.13it/s, loss=-0.907015]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

MLE fitting:  15%|█▌        | 1501/10000 [00:01<00:09, 888.71it/s, loss=-0.954419]


2PL abilities:        min = -1.39, max = 0.56
2PL difficulties:     min = -1.30, max = 3.75
2PL discriminations:  min = 0.17, max = 3.30, mean = 1.00
discriminations_2pl: Updated existing row  cohort='Q4-2024'  arch='Gemma2ForCausalLM'
difficulties_2pl: Updated existing row  cohort='Q4-2024'  arch='Gemma2ForCausalLM'

  arch=Gemma2ForCausalLM  cohort=Q1-2025
Response matrix: 37 models × 24 items


MLE fitting:  14%|█▍        | 1400/10000 [00:01<00:06, 1317.87it/s, loss=-0.936445]


2PL abilities:        min = -1.03, max = 0.45
2PL difficulties:     min = -1.31, max = 2.60
2PL discriminations:  min = 0.14, max = 2.63, mean = 0.98
discriminations_2pl: Updated existing row  cohort='Q1-2025'  arch='Gemma2ForCausalLM'
difficulties_2pl: Updated existing row  cohort='Q1-2025'  arch='Gemma2ForCausalLM'
